### Tables Cleanup

In [0]:
# Step 1: List all tables in hls_glucosphere.cgm
tables_df = spark.sql("SHOW TABLES IN hls_glucosphere.cgm")
table_list = [row.tableName for row in tables_df.collect()]

print(f"Found {len(table_list)} tables in hls_glucosphere.cgm:")
for table in sorted(table_list):
    print(f"  - {table}")

# Separate diabetes_* tables from others
diabetes_tables = [t for t in table_list if t.startswith('diabetes_')]
non_diabetes_tables = [t for t in table_list if not t.startswith('diabetes_')]

print(f"\n📊 Summary:")
print(f"  Total tables: {len(table_list)}")
print(f"  Diabetes tables (to preserve): {len(diabetes_tables)}")
print(f"  Non-diabetes tables (to drop): {len(non_diabetes_tables)}")

In [0]:
# Step 2: Create cgm_devs schema and copy all tables
print("Creating schema hls_glucosphere.cgm_devs if not exists...")
spark.sql("CREATE SCHEMA IF NOT EXISTS hls_glucosphere.cgm_devs")
print("✓ Schema ready\n")

# Check existing tables in cgm_devs
existing_tables_df = spark.sql("SHOW TABLES IN hls_glucosphere.cgm_devs")
existing_tables = [row.tableName for row in existing_tables_df.collect()]

print(f"Copying {len(table_list)} tables from cgm to cgm_devs...\n")

copied = []
skipped = []
errors = []

for table in table_list:
    source = f"hls_glucosphere.cgm.{table}"
    target = f"hls_glucosphere.cgm_devs.{table}"
    
    if table in existing_tables:
        print(f"  ⏭️ {table}: Already exists (skipped)")
        skipped.append(table)
    else:
        try:
            spark.sql(f"CREATE TABLE {target} AS SELECT * FROM {source}")
            count = spark.sql(f"SELECT COUNT(*) as cnt FROM {target}").collect()[0].cnt
            print(f"  ✓ {table}: {count:,} rows copied")
            copied.append(table)
        except Exception as e:
            print(f"  ✗ {table}: ERROR - {str(e)}")
            errors.append(table)

print(f"\n📊 Summary:")
print(f"  Copied: {len(copied)} tables")
print(f"  Skipped (already exist): {len(skipped)} tables")
print(f"  Errors: {len(errors)} tables")

In [0]:
# Step 3: List and copy MLflow registered models
print("Searching for registered models in hls_glucosphere.cgm...\n")

try:
    # Query information schema for models
    models_df = spark.sql("""
        SELECT model_name, model_catalog, model_schema
        FROM hls_glucosphere.information_schema.models
        WHERE model_schema = 'cgm'
    """)
    
    models_list = models_df.collect()
    
    if not models_list or len(models_list) == 0:
        print("ℹ️ No registered models found in hls_glucosphere.cgm")
    else:
        print(f"Found {len(models_list)} registered model(s):\n")
        
        for row in models_list:
            full_name = f"{row.model_catalog}.{row.model_schema}.{row.model_name}"
            print(f"  🧠 {full_name}")
        
        print("\n⚠️  Note: To copy models to cgm_devs, use SQL commands like:")
        print("    CREATE MODEL hls_glucosphere.cgm_devs.<model_name>")
        print("    FROM hls_glucosphere.cgm.<model_name> VERSION <version>")
        
except Exception as e:
    error_msg = str(e)
    if "SCHEMA_NOT_FOUND" in error_msg or "cannot be found" in error_msg or "TABLE_OR_VIEW_NOT_FOUND" in error_msg:
        print("ℹ️ No registered models found in hls_glucosphere.cgm")
    else:
        print(f"ℹ️ Could not query registered models: {error_msg[:150]}...")
    print("This is expected if no models are registered in this schema.")

In [0]:
# Step 4: Summary and cleanup
print("="*70)
print("MIGRATION SUMMARY")
print("="*70)

print(f"\n✓ Tables in hls_glucosphere.cgm: {len(table_list)}")
for table in sorted(table_list):
    print(f"  - {table}")

print(f"\n✓ Tables copied to hls_glucosphere.cgm_devs: {len(table_list)}")
print(f"\n✓ Registered models found: 0")

print(f"\n📊 Analysis:")
print(f"  - Diabetes tables (to preserve): {len(diabetes_tables)}")
for table in sorted(diabetes_tables):
    print(f"    • {table}")

print(f"\n  - Non-diabetes tables (to drop): {len(non_diabetes_tables)}")
if non_diabetes_tables:
    for table in sorted(non_diabetes_tables):
        print(f"    • {table}")
    print("\n" + "="*70)
    print("SQL COMMANDS TO DROP NON-DIABETES TABLES:")
    print("="*70)
    for table in non_diabetes_tables:
        print(f"DROP TABLE IF EXISTS hls_glucosphere.cgm.{table};")
    print("\n⚠️  Execute the DROP statements above to complete the cleanup.")
else:
    print("    (None - all tables start with diabetes_*)")
    print("\n✓ No cleanup needed! All tables in cgm are diabetes_* tables.")

print("\n" + "="*70)
print("✓ MIGRATION COMPLETE")
print("="*70)

### Models Cleanup

In [0]:
# List all models in hls_glucosphere.cgm and generate delete commands
import mlflow
from mlflow.tracking import MlflowClient

client = MlflowClient()
mlflow.set_registry_uri('databricks-uc')

CATALOG = "hls_glucosphere"
SCHEMA = "cgm"
prefix = f"{CATALOG}.{SCHEMA}."

print(f"Searching for models in {CATALOG}.{SCHEMA}...\n")

try:
    # Get all models and filter manually (Unity Catalog doesn't support filter_string)
    all_models = client.search_registered_models(max_results=1000)
    
    # Filter to only models in our schema
    models = [m for m in all_models if m.name.startswith(prefix)]
    
    if not models:
        print(f"✓ No models found in {CATALOG}.{SCHEMA}")
    else:
        print(f"Found {len(models)} model(s):\n")
        
        delete_commands = []
        total_versions = 0
        
        for model in models:
            model_name = model.name
            print(f"🧠 {model_name}")
            
            # Get all versions
            versions = client.search_model_versions(f"name='{model_name}'")
            version_count = len(versions)
            total_versions += version_count
            print(f"   Versions: {version_count}")
            
            for version in versions:
                print(f"     - Version {version.version} (status: {version.status})")
                delete_commands.append(f"client.delete_model_version(name='{model_name}', version='{version.version}')")
            
            delete_commands.append(f"client.delete_registered_model(name='{model_name}')")
            print()
        
        print("="*70)
        print("DELETE COMMANDS (execute these to remove models):")
        print("="*70)
        print("\n# Delete all versions and models:")
        for cmd in delete_commands:
            print(cmd)
        
        print("\n" + "="*70)
        print("SUMMARY")
        print("="*70)
        print(f"Total models: {len(models)}")
        print(f"Total versions: {total_versions}")
        print("\n⚠️  Copy and execute the delete commands above to remove all models.")
        
except Exception as e:
    print(f"Error: {str(e)}")
    print(f"\nℹ️ No models found or unable to access models in {CATALOG}.{SCHEMA}")

In [0]:
# Execute deletion of all models and versions
# WARNING: This will permanently delete all models in hls_glucosphere.cgm

import mlflow
from mlflow.tracking import MlflowClient

client = MlflowClient()
mlflow.set_registry_uri('databricks-uc')

CATALOG = "hls_glucosphere"
SCHEMA = "cgm"
prefix = f"{CATALOG}.{SCHEMA}."

print("Starting deletion process...\n")
print(f"Target: {CATALOG}.{SCHEMA}\n")

try:
    # Get all models and filter to our schema
    all_models = client.search_registered_models(max_results=1000)
    models = [m for m in all_models if m.name.startswith(prefix)]
    
    if not models:
        print(f"✓ No models found in {CATALOG}.{SCHEMA}")
    else:
        print(f"Found {len(models)} model(s) to delete\n")
        
        deleted_models = 0
        deleted_versions = 0
        errors = []
        
        for model in models:
            model_name = model.name
            print(f"🧠 Processing: {model_name}")
            
            try:
                # Get all versions for this model
                versions = client.search_model_versions(f"name='{model_name}'")
                print(f"   Found {len(versions)} version(s)")
                
                # Delete all versions first
                for version in versions:
                    try:
                        client.delete_model_version(name=model_name, version=version.version)
                        print(f"   ✓ Deleted version {version.version}")
                        deleted_versions += 1
                    except Exception as e:
                        error_msg = f"Failed to delete {model_name} version {version.version}: {str(e)}"
                        print(f"   ✗ {error_msg}")
                        errors.append(error_msg)
                
                # Delete the model itself
                try:
                    client.delete_registered_model(name=model_name)
                    print(f"   ✓ Deleted model {model_name}\n")
                    deleted_models += 1
                except Exception as e:
                    error_msg = f"Failed to delete model {model_name}: {str(e)}"
                    print(f"   ✗ {error_msg}\n")
                    errors.append(error_msg)
                    
            except Exception as e:
                error_msg = f"Failed to process model {model_name}: {str(e)}"
                print(f"   ✗ {error_msg}\n")
                errors.append(error_msg)
        
        print("="*70)
        if errors:
            print("⚠️  DELETION COMPLETED WITH ERRORS")
        else:
            print("✓ ALL MODELS DELETED SUCCESSFULLY")
        print("="*70)
        print(f"\nDeleted:")
        print(f"  - {deleted_models} model(s)")
        print(f"  - {deleted_versions} version(s)")
        
        if errors:
            print(f"\nErrors ({len(errors)}):")
            for error in errors:
                print(f"  ✗ {error}")
        
except Exception as e:
    print(f"\n✗ Fatal error during deletion: {str(e)}")
    print("Process terminated.")